<a href="https://colab.research.google.com/github/the-visible-ghost/ir26/blob/main/notebooks/sandbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# PhantomDevs - Sandbox - India Runs 2026
> **Note:** Downloading models require internet access

In [4]:
# General Typing imports
from typing import (
    List,
    Dict,
    Literal,
    Optional,
    Tuple,
)
from time import perf_counter

In [5]:
# Global Constants ---
DEBUG: bool = True
CANDIDATES_FILE: str = "./candidates.jsonl" # Or "./candidates.jsonl.gz"
OUTPUT_PATH: str = "./submission.csv"

VOWELS = ("a", "e", "i", "o", "u")

# Model Names ---
MODELS: Dict = {
    "embedder": "BAAI/bge-small-en-v1.5",
    "reranker": "cross-encoders/ms-marco-MiniLM-L-12-v2"
}

# Preprocessed directory path ---
PROCESSED_DIR: str = "./processed"

# Processed filenames ---
PROCESSED_FILENAMES: Dict[
    Literal["job_desc", "indexes", "vectors", "lookup"],
    Dict[Literal["summary", "skills", "experience"], str] | str
] = {
    "job_desc": PROCESSED_DIR + "/job_desc.json",
    "indexes": {
        "summary": PROCESSED_DIR + "/indexes/summary.faiss",
        "skills": PROCESSED_DIR + "/indexes/skills.faiss",
        "experience": PROCESSED_DIR + "/indexes/experience.faiss",
    },
    "vectors": {
        "summary": PROCESSED_DIR + "/vectors/summary.npy",
        "skills": PROCESSED_DIR + "/vectors/skills.npy",
        "experience": PROCESSED_DIR + "/vectors/experience.npy",
    },
    "lookup": {
        "summary": PROCESSED_DIR + "/lookup/summary.json",
        "skills": PROCESSED_DIR + "/lookup/skills.json",
        "experience": PROCESSED_DIR + "/lookup/experience.json",
    }
}

# Scoring Weights ---
OVERALL_WEIGHTS = {
    "redrob": 0.01,
    "penalty": -1, # for Inconsistent Profiles (Honeypots)
    "match": 1,
    "reranker": 1,
}

CANDIDATE_PENALTY = {
    "skills_experience_mismatch": 1,
    "summary_experience_mismatch": 0.66,
    "summary_skills_mismatch": 0.33,
}

CATEGORY_WEIGHTS = {
    "must_have": 5.0,
    "core_ml_retrieval": 4.0,
    "evaluation_ml_depth": 4.0,
    "execution_signal": 3.0,
    "nice_to_have": 1.5,
    "context": 1.0,
    "negative_signals": -4.0, # Filtering out JD's "DO NOT WANT" candidates
}

SECTION_WEIGHTS = {
    "skills": 1.5,
    "experience": 1.2,
    "summary": 1.0,
}

REDROB_SIGNAL_WEIGHTS = {
    "profile_completeness_score": 0.25,
    "verified_email": 0.20,
    "verified_phone": 0.25,
    "linkedin_connected": 0.15,
    "open_to_work_flag": 0.50,
    "notice_period_days": -0.20,
    "profile_views_received_30d": 0.10,
    "search_appearance_30d": 0.10,
    "saved_by_recruiters_30d": 0.30,
    "applications_submitted_30d": 0.15,
    "recruiter_response_rate": 0.45,
    "avg_response_time_hours": -0.35,
    "github_activity_score": 0.35,
    "endorsements_received": 0.15,
    "interview_completion_rate": 0.40,
    "offer_acceptance_rate": 0.35,
    "willing_to_relocate": 0.20,
}

# General configurations and declarations

In [6]:
!pip install msgspec faiss-cpu tqdm
import msgspec, faiss, json, os
import numpy as np
from tqdm import tqdm

# Creating directories ----
def extract_paths(d):
    paths = []
    for v in d.values():
        if not isinstance(v, dict):
            paths.append(v); continue
        paths.extend(extract_paths(v))
    return paths

for path in tqdm(extract_paths(PROCESSED_FILENAMES), desc="Creating directories"):
    os.makedirs(os.path.dirname(path), exist_ok=True)

Creating directories: 100%|██████████| 10/10 [00:00<00:00, 11997.44it/s]


### Debug logs helper functions and wrapper delcarations

In [7]:
# General Utility Functions and Wrappers ---
def debug_print(*args, **kwargs):
    if DEBUG: print("[DEBUG]:", *args, **kwargs)


def debug(func):
    def wrapper(*args, **kwargs):
        debug_print(f"{func.__qualname__}({'...' if args or kwargs else ''}) called")
        start = perf_counter()
        retval = func(*args, **kwargs)
        elapsed = perf_counter() - start
        debug_print(
            f"{func.__qualname__}({'...' if args or kwargs else ''}) "
            f"{'finished' if retval is None else 'returned'} in {elapsed:.3f} s"
        )
        return retval
    return wrapper

### JSON helpers

In [8]:
@debug
def load_json_file(file: str) -> Dict | List:
    debug_print(f"Loading from \"{file}\"")
    with open(file, "r") as fp:
        return json.load(fp)


@debug
def dump_json_file(data: Dict | List, file: str) -> None:
    debug_print(f"Dumping in \"{file}\"")
    with open(file, "w") as fp:
        json.dump(data, fp)

### Skill Cluster Configuration delcarations

In [9]:
CLUSTER_DESCRIPTIONS = {
    "software_engineering": "Experience building software systems and production applications.",
    "data_engineering": "Experience building data pipelines and large-scale data infrastructure.",
    "retrieval_ranking": "Experience with search, retrieval, recommendation and ranking systems.",
    "machine_learning": "Experience applying machine learning models and statistical techniques.",
    "deep_learning": "Experience with deep learning and computer vision systems.",
    "nlp_llm": "Experience with NLP systems, language models and fine-tuning techniques.",
    "mlops": "Experience deploying, serving and monitoring machine learning systems.",
    "cloud": "Experience with cloud infrastructure and distributed systems deployment.",
    "database": "Experience designing and working with database and storage systems.",
    "frontend": "Experience building user interfaces and frontend applications.",
    "domain": "Experience in domain-specific business and industry applications.",
    "leadership": "Experience leading teams, projects and engineering processes.",
}

SKILL_CLUSTER_MAP = {
    "software_engineering": [
        "Python",
        "Java",
        "JavaScript",
        "TypeScript",
        "Go",
        "Rust",
        "Node.js",
        "Django",
        "Flask",
        "FastAPI",
        "Spring Boot",
        "gRPC",
        "REST APIs",
        "Microservices",
        "GraphQL",
        "Next.js",
        "React",
        "Redux",
        "Vue.js",
        "HTML",
        "CSS",
        "Tailwind",
        "Webpack",
    ],
    "data_engineering": [
        "Spark",
        "Airflow",
        "Kafka",
        "Apache Flink",
        "Apache Beam",
        "Databricks",
        "ETL",
        "Data Pipelines",
        "dbt",
        "Snowflake",
        "BigQuery",
        "Hadoop",
    ],
    "retrieval_ranking": [
        "BM25",
        "Elasticsearch",
        "OpenSearch",
        "FAISS",
        "Pinecone",
        "Weaviate",
        "Qdrant",
        "Milvus",
        "Vector Search",
        "Recommendation Systems",
        "Information Retrieval",
        "Haystack",
    ],
    "machine_learning": [
        "Feature Engineering",
        "XGBoost",
        "Statistical Modeling",
        "Forecasting",
        "scikit-learn",
        "Machine Learning",
        "Data Science",
    ],
    "deep_learning": [
        "PyTorch",
        "TensorFlow",
        "CNN",
        "GANs",
        "Object Detection",
        "Image Classification",
        "YOLO",
        "OpenCV",
        "Computer Vision",
        "Deep Learning",
        "Reinforcement Learning",
    ],
    "nlp_llm": [
        "NLP",
        "Hugging Face Transformers",
        "Fine-tuning LLMs",
        "LoRA",
        "PEFT",
        "Prompt Engineering",
        "LangChain",
        "Sentence Transformers",
        "Embeddings",
        "Speech Recognition",
        "TTS",
    ],
    "mlops": [
        "BentoML",
        "MLflow",
        "Kubeflow",
        "Weights & Biases",
        "MLOps",
        "CI/CD",
    ],
    "cloud": [
        "AWS",
        "Azure",
        "GCP",
        "Kubernetes",
        "Docker",
        "Terraform",
    ],
    "database": [
        "SQL",
        "PostgreSQL",
        "MongoDB",
        "Redis",
    ],
    "frontend": [
        "Angular",
        "Figma",
        "Illustrator",
        "Photoshop",
    ],
    "domain": [
        "Marketing",
        "Sales",
        "Salesforce CRM",
        "Accounting",
        "Tally",
        "SAP",
        "Content Writing",
        "SEO",
        "Excel",
        "PowerPoint",
    ],
    "leadership": [
        "Scrum",
        "Agile",
        "Project Management",
        "Six Sigma",
    ],
}

SKILL_TO_CLUSTER = {}
for cluster, skills in SKILL_CLUSTER_MAP.items():
    for skill in skills:
        SKILL_TO_CLUSTER[skill] = cluster


def gen_skill_cluster(candidate):
    clusters = defaultdict(list)
    for skill in candidate.skills:
        cluster = SKILL_TO_CLUSTER.get(skill.name)
        if not cluster: continue
        clusters[cluster].append(skill)
    return dict(clusters)

### Candidate data wrapper objects and types declarations

In [38]:
import datetime as dt

type CompanySize = Literal[
    "1-10",
    "11-50",
    "51-200",
    "201-500",
    "501-1000",
    "1001-5000",
    "5001-10000",
    "10001+",
]

type EducationTier = Literal["tier_1", "tier_2", "tier_3", "tier_4", "unknown"]
type SkillProficiency = Literal["beginner", "intermediate", "advanced", "expert"]
type LanguageProficiency = Literal["basic", "conversational", "professional", "native"]
type WorkMode = Literal["remote", "hybrid", "onsite", "flexible"]

type CandidateId = str
type CandidateEmbedData = Dict[
    Literal["summary", "skills", "experience"], str | List[str]
]

class ReasonFeatures:
    def __init__(self):
        # Match quality
        self.match_score: float = 0.0
        self.match_category: str = "low"  # low, medium, high
        self.top_matching_sections: List[str] = []

        # Career analysis
        self.career_trajectory: str = "unknown"  # stable, gap, pivot, transition, fresh
        self.yoe: float = 0.0
        self.current_title: str = ""
        self.current_company: str = ""
        self.current_industry: str = ""
        self.past_titles: List[str] = []
        self.past_industries: List[str] = []
        self.has_relevant_current_role: bool = False
        self.has_relevant_past_role: bool = False

        # Skills analysis
        self.relevant_skill_count: int = 0
        self.advanced_skill_count: int = 0
        self.skill_source: str = "unknown"  # current_job, past_job, self_taught, mixed
        self.top_skills: List[str] = []
        self.skill_assessment_avg: Optional[float] = None

        # Signals
        self.profile_completeness: float = 0.0
        self.is_verified: bool = False
        self.is_active: bool = False
        self.open_to_work: bool = False
        self.response_rate: float = 0.0
        self.github_active: bool = False

        # Mismatch / penalty analysis
        self.internal_penalty: float = 0.0
        self.mismatch_type: str = (
            "none"  # none, minor, career_pivot, skill_inflation, honeypot_like
        )
        self.penalty_breakdown: Dict[str, float] = {}

        # Temporal
        self.career_gap_months: int = 0
        self.is_recent_graduate: bool = False
        self.is_career_changer: bool = False
        self.years_since_last_relevant: float = 0.0


class Profile(msgspec.Struct):
    anonymized_name: str
    headline: str
    summary: str
    location: str
    country: str
    years_of_experience: float
    current_title: str
    current_company: str
    current_company_size: CompanySize
    current_industry: str

class Career(msgspec.Struct):
    company: str
    title: str
    start_date: dt.date
    end_date: Optional[dt.date]
    duration_months: int
    is_current: bool
    industry: str
    company_size: CompanySize
    description: str

    @property
    def embed_str(self) -> str:
        return (
            f"I {'am' if self.is_current else 'was'} "
            f"{'currently' if self.is_current else 'previously'} "
            f"{'an' if self.title[0] in vowels else 'a'} {self.title} "
            f"at {self.company} ({self.company_size} employees) "
            f"in {self.industry} industry for "
            f"{self.duration_months // 12} years.\n" + self.description
        )

class Education(msgspec.Struct):
    institution: str
    degree: str
    field_of_study: str
    start_year: int
    end_year: int
    grade: Optional[str]
    tier: EducationTier

class Skill(msgspec.Struct):
    name: str
    proficiency: SkillProficiency
    endorsements: int
    duration_months: int

    @property
    def embed_str(self) -> str:
        return (
            f"{self.proficiency} proficiency in "
            f"{self.name} with "
            f"{self.duration_months // 12} years experience"
        )

class SkillCluster(msgspec.Struct):
    name: str
    desc: str
    skills: List[Skill]

    @property
    def embed_str(self) -> str:
        skills = "\n".join(" - " + skill.embed_str for skill in self.skills)
        return f"{self.name}:\n{self.desc}\n\n{skills}"

class Certification(msgspec.Struct):
    name: str
    issuer: str
    year: int

class Language(msgspec.Struct):
    language: str
    proficiency: LanguageProficiency

class RedrobSignals(msgspec.Struct):
    profile_completeness_score: float
    signup_date: dt.date
    last_active_date: dt.date
    open_to_work_flag: bool
    profile_views_received_30d: int
    applications_submitted_30d: int
    recruiter_response_rate: float
    avg_response_time_hours: float
    skill_assessment_scores: Dict[str, float]
    connection_count: int
    endorsements_received: int
    notice_period_days: int
    expected_salary_range_inr_lpa: Dict[Literal["min", "max"], float]
    preferred_work_mode: WorkMode
    willing_to_relocate: bool
    github_activity_score: float
    search_appearance_30d: int
    saved_by_recruiters_30d: int
    interview_completion_rate: float
    offer_acceptance_rate: float
    verified_email: bool
    verified_phone: bool
    linkedin_connected: bool

class Candidate(msgspec.Struct):
    candidate_id: CandidateId
    profile: Profile
    career_history: List[Career]
    education: List[Education]
    skills: List[Skill]
    certifications: List[Certification]
    languages: List[Language]
    redrob_signals: RedrobSignals

    _reason_features: Optional[ReasonFeatures] = None

    @property
    def skill_clusters(self) -> Dict[str, SkillCluster]:
        return {
            name: SkillCluster(name, CLUSTER_DESCRIPTIONS[name], skills)
            for name, skills in gen_skill_cluster(self).items()
        }

    @property
    def embed_data(self) -> CandidateEmbedData:
        return {
            "summary": (
                f"I am a {self.profile.current_title}"
                f" at {self.profile.current_company}"
                f" in {self.profile.current_industry} industry."
                f" {self.profile.summary}"
            ),
            "skills": [cluster.embed_str for cluster in self.skill_clusters.values()],
            "experience": [carrer.embed_str for carrer in self.career_history],
        }

    @property
    def text(self) -> str:
        embed_data = self.embed_data
        return (
            f"Sumamary: {embed_data['summary']}. "
            f"Skills: {', '.join(skill.embed_str for skill in self.skills)}. "
            f"Experience: {', '.join(embed_data['experience'])}."
        )

    # REASON GENERATION -------------------------------------------------------------

    def _analyze_career_trajectory(self) -> Tuple[str, Dict]:
        career = self.career_history
        if not career:
            return "unknown", {}

        details = {
            "current_title": career[0].title if career else "",
            "past_titles": [c.title for c in career[1:]],
            "current_industry": career[0].industry if career else "",
            "past_industries": list(set(c.industry for c in career[1:])),
            "total_months": sum(c.duration_months for c in career),
            "gap_months": 0,
            "is_fresh": self.profile.years_of_experience < 2 and len(career) <= 2,
        }

        # Detect career gaps between consecutive jobs
        if len(career) >= 2:
            for i in range(len(career) - 1):
                current_end = career[i].end_date
                next_start = career[i + 1].start_date
                if current_end and next_start:
                    gap = (next_start - current_end).days / 30
                    if gap > 3:  # > 3 months gap
                        details["gap_months"] += int(gap)

        # Detect pivot: current role very different from past roles
        if len(career) >= 2:
            current_title_lower = career[0].title.lower()
            past_titles_lower = [c.title.lower() for c in career[1:]]

            # Simple keyword overlap for pivot detection
            current_keywords = set(current_title_lower.split())
            pivot_score = 0
            for past in past_titles_lower:
                past_keywords = set(past.split())
                overlap = len(current_keywords & past_keywords)
                union = len(current_keywords | past_keywords)
                if union > 0:
                    pivot_score += overlap / union

            avg_similarity = (
                pivot_score / len(past_titles_lower) if past_titles_lower else 1.0
            )

            if avg_similarity < 0.3 and details["gap_months"] > 6:
                return "pivot_with_gap", details
            elif avg_similarity < 0.3:
                return "pivot", details
            elif details["gap_months"] > 6:
                return "gap", details

        if details["is_fresh"]:
            return "fresh", details

        return "stable", details

    def _analyze_skill_sources(self) -> Tuple[str, List[str]]:
        career = self.career_history
        skills = self.skills

        if not skills or not career:
            return "unknown", []

        # Build text from all career entries
        all_exp_text = " ".join(f"{c.title} {c.description}".lower() for c in career)
        current_exp_text = (
            f"{career[0].title} {career[0].description}".lower() if career else ""
        )
        past_exp_text = " ".join(
            f"{c.title} {c.description}".lower() for c in career[1:]
        )

        current_matches = []
        past_matches = []
        unmatched = []

        for skill in skills:
            skill_name = skill.name.lower()
            skill_words = set(skill_name.split()) | {skill_name}

            # Check if skill appears in current role
            in_current = any(word in current_exp_text for word in skill_words)
            in_past = any(word in past_exp_text for word in skill_words)
            in_any = any(word in all_exp_text for word in skill_words)

            if in_current:
                current_matches.append(skill.name)
            elif in_past:
                past_matches.append(skill.name)
            elif in_any:
                past_matches.append(skill.name)
            else:
                unmatched.append(skill.name)

        # Determine primary source
        total = len(skills)
        current_ratio = len(current_matches) / total if total > 0 else 0
        past_ratio = len(past_matches) / total if total > 0 else 0
        unmatched_ratio = len(unmatched) / total if total > 0 else 0

        if current_ratio >= 0.6:
            return "current_job", current_matches
        elif past_ratio >= 0.5 and current_ratio < 0.3:
            return "past_job", past_matches
        elif unmatched_ratio >= 0.5:
            return "self_taught", unmatched
        else:
            return "mixed", current_matches + past_matches

    def _classify_mismatch(
        self, penalty: float, penalty_breakdown: Dict[str, float]
    ) -> str:
        if penalty < 0.2:
            return "none"

        # Check which penalties are high
        high_penalties = {k: v for k, v in penalty_breakdown.items() if v > 0.5}

        if not high_penalties:
            return "minor"

        # Career pivot: high title-exp mismatch but career is coherent
        if (
            "title_exp_penalty" in high_penalties
            and "career_penalty" not in high_penalties
        ):
            trajectory, _ = self._analyze_career_trajectory()
            if trajectory in ("pivot", "pivot_with_gap"):
                return "career_pivot"

        # Skill inflation: many advanced skills but low assessment
        if "skill_inflation_penalty" in high_penalties:
            assessments = self.redrob_signals.skill_assessment_scores
            if assessments:
                avg = np.mean(list(assessments.values()))
                if avg < 50:
                    return "honeypot_like"
            # Could be genuine expert with no assessments
            return "skill_expert"

        # Summary doesn't match experience: possible fake profile
        if (
            "summary_exp_penalty" in high_penalties
            and "skill_exp_penalty" in high_penalties
        ):
            return "honeypot_like"

        # Education issues
        if "education_penalty" in high_penalties:
            return "data_quality_issue"

        return "minor"

    def _build_reason(self, features: ReasonFeatures) -> str:
        p = self.profile
        r = self.redrob_signals
        career = self.career_history

        # Determine what makes this candidate stand out
        standout = []

        # 1. Relevant experience depth
        if features.match_score >= 0.75:
            standout.append("deep role alignment")
        elif features.match_score >= 0.5:
            standout.append("relevant background")

        # 2. Proven skills from actual work
        past_skills = [s.name for s in self.skills if s.duration_months >= 24]
        if len(past_skills) >= 2:
            standout.append(f"proven {past_skills[0]} and {past_skills[1]} expertise")
        elif len(past_skills) == 1:
            standout.append(f"proven {past_skills[0]} expertise")

        # 3. Current applicability
        if features.has_relevant_current_role:
            standout.append("actively working in a similar role")
        elif features.has_relevant_past_role and career:
            relevant_past = [c for c in career[1:] if not c.is_current]
            if relevant_past:
                standout.append(f"prior {relevant_past[0].title} experience")

        # 4. Engagement signals
        if r.open_to_work_flag and r.recruiter_response_rate > 0.5:
            standout.append("responsive and actively looking")
        elif r.open_to_work_flag:
            standout.append("open to opportunities")

        # Build the sentence
        parts = []

        # Identity clause
        identity = f"{p.years_of_experience:.0f}-year {p.current_title}"
        if career and len(career) > 1:
            past = career[1].title
            identity += f" (previously {past})"

        parts.append(identity)

        # Standout clause
        if standout:
            parts.append(f"brings {', '.join(standout)}")

        # Caveat clause (only if notable)
        caveat = ""
        trajectory, traj_details = self._analyze_career_trajectory()
        mismatch = self._classify_mismatch(
            features.internal_penalty, features.penalty_breakdown
        )

        if mismatch == "career_pivot" and traj_details.get("gap_months", 0) > 12:
            caveat = f"career gap of {traj_details['gap_months'] // 12} years"
        elif mismatch == "honeypot_like":
            caveat = "claims need verification"
        elif not (r.verified_email and r.verified_phone):
            caveat = "contact details unverified"
        elif not r.open_to_work_flag:
            caveat = "passive candidate"

        if caveat:
            parts.append(f"note: {caveat}")

        reason = ", ".join(parts) + "."
        return reason[:247] + "..." if len(reason) > 250 else reason

    @property
    def reason(self) -> str:
        if self._reason_features is not None:
            return self._build_reason(self._reason_features)

        # Fallback: basic reason without scoring context
        p = self.profile
        r = self.redrob_signals

        # Count AI-relevant skills
        ai_clusters = (
            "retrieval_ranking",
            "machine_learning",
            "nlp_llm",
            "deep_learning",
        )
        ai_skills = sum(
            len(cluster.skills)
            for name, cluster in self.skill_clusters.items()
            if name in ai_clusters
        )

        # Basic trajectory check
        trajectory, _ = self._analyze_career_trajectory()

        if trajectory == "fresh":
            return (
                f"{p.years_of_experience:.1f}YOE {p.current_title} "
                f"with {ai_skills} AI core skills; "
                f"response rate {r.recruiter_response_rate:.0%}."
            )
        elif trajectory in ("pivot", "pivot_with_gap"):
            past = (
                self.career_history[1].title
                if len(self.career_history) > 1
                else "professional"
            )
            return (
                f"Former {past} ({p.years_of_experience:.1f}YOE), "
                f"currently {p.current_title}; "
                f"{ai_skills} AI skills; "
                f"response rate {r.recruiter_response_rate:.0%}."
            )
        else:
            return (
                f"Currently {p.current_title} "
                f"with {p.years_of_experience:.1f}YOE; "
                f"{ai_skills} AI core skills; "
                f"response rate {r.recruiter_response_rate:.0%}."
            )

# Pre-Processing

## Resolving Dependencies

### For GPU Preprocessing (Recommended)

In [11]:
!pip uninstall -y fastembed onnxruntime-gpu onnxruntime
!pip install onnxruntime-gpu==1.21.1 fastembed-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.8/280.8 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.1 MB/s eta 0:00:00


### For CPU Preprocessing

In [ ]:
!pip uninstall -y onnxruntime-gpu fastembed-gpu
!pip install fastembed

Found existing installation: onnxruntime-gpu 1.21.1
Uninstalling onnxruntime-gpu-1.21.1:
  Successfully uninstalled onnxruntime-gpu-1.21.1
Found existing installation: fastembed-gpu 0.8.0
Uninstalling fastembed-gpu-0.8.0:
  Successfully uninstalled fastembed-gpu-0.8.0
  Using cached fastembed-0.8.0-py3-none-any.whl.metadata (10 kB)
  Using cached onnxruntime-1.27.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.4 kB)
Using cached fastembed-0.8.0-py3-none-any.whl (116 kB)
Using cached onnxruntime-1.27.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (18.7 MB)


## Setting up Embedder Model

In [12]:
from fastembed import TextEmbedding
from collections import defaultdict

class Embedder:
    def __init__(self, name: str) -> None:
        self._embedder: TextEmbedding = TextEmbedding(name)

    @debug
    def embed(self, texts: str | List[str]) -> np.ndarray:
        return np.array(list(self._embedder.embed(texts)))

embedder = Embedder(MODELS["embedder"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

## Pre-Processing Job Description

### The Processed JD
Atomically written JD's components in categories.

You must not change the category names such as "must_have", "core_ml_retrieval", etc.

Change only the strings inside each categories, you can put different number of
strings in each categories but do not overpopulate categories.

Do not write the components over-atomically, and do not make each component (string) noisy.

In [13]:
jd_processed = {
    "must_have": [
        "Senior AI Engineer role focused on production ML systems for ranking, retrieval, and candidate-job matching in a talent intelligence platform.",
        "Strong production experience building embeddings-based retrieval systems using models like sentence-transformers, BGE, E5, or OpenAI embeddings, deployed in real user-facing systems.",
        "Hands-on experience designing and operating vector search or hybrid retrieval systems using FAISS, Milvus, Weaviate, Pinecone, Elasticsearch, or OpenSearch.",
        "Strong Python engineering skills with production-grade software development practices and ownership of deployed ML systems.",
        "Experience designing and owning ranking evaluation frameworks including NDCG, MRR, MAP, offline-to-online evaluation, and A/B testing for production ranking systems.",
        "Proven experience building and shipping end-to-end ranking, search, recommendation, or retrieval systems that are deployed to real users.",
        "Experience owning production ML retrieval systems including embedding drift handling, index refresh strategies, and monitoring retrieval quality regression.",
    ],
    "core_ml_retrieval": [
        "Deep expertise in modern ML retrieval systems including embeddings, dense retrieval, hybrid retrieval, ranking systems, and recommendation architectures.",
        "Experience designing and scaling large-scale retrieval pipelines with attention to latency, recall, precision, and production constraints.",
        "Understanding of tradeoffs between BM25 keyword search, dense embedding retrieval, and hybrid retrieval systems.",
        "Experience with learning-to-rank systems and ranking optimization strategies in production.",
        "Experience building distributed or large-scale ML inference systems for search and retrieval workloads.",
    ],
    "evaluation_ml_depth": [
        "Strong ability to design evaluation systems for ranking and retrieval including offline evaluation pipelines and online A/B testing systems.",
        "Deep understanding of ranking metrics such as NDCG, MRR, MAP, precision-recall tradeoffs, and ranking correlation metrics.",
        "Experience building feedback loops from user behavior into ranking system improvements in production systems.",
        "Ability to design statistically sound experiments and interpret ranking system improvements rigorously.",
        "Strong intuition for measurement, experimentation, and iterative ML system improvement.",
    ],
    "nice_to_have": [
        "Experience with LLM fine-tuning techniques such as LoRA, QLoRA, or PEFT.",
        "Familiarity with retrieval-augmented generation (RAG) systems and hybrid LLM + retrieval pipelines.",
        "Experience with learning-to-rank models including XGBoost-based rankers or neural ranking models.",
        "Exposure to HR-tech, recruiting platforms, or marketplace matching systems.",
        "Background in distributed systems, large-scale data pipelines, or inference optimization.",
        "Open-source contributions in machine learning, search, or information retrieval systems.",
    ],
    "execution_signal": [
        "Ability to rapidly ship production systems in fast-moving environments, starting from imperfect implementations and iterating based on real user feedback.",
        "Strong bias toward execution over research purity, with ability to build working v1 ranking systems quickly.",
        "Experience working closely with product and operations teams to refine ML systems in production.",
        "Comfort operating in ambiguous environments where requirements evolve continuously.",
        "Preference for practical engineering solutions over theoretical completeness in early-stage systems.",
    ],
    "negative_signals": [
        "Career limited exclusively to IT consulting companies such as TCS, Infosys, Wipro, Accenture, Cognizant, or Capgemini without product ML experience.",
        "Pure research-only background without production deployment of ML or ranking systems.",
        "Recent experience limited to framework-based AI development such as LangChain wrappers without deeper ML system ownership.",
        "Primary expertise in computer vision, speech, or robotics without significant NLP or information retrieval experience.",
        "Senior-level roles focused only on architecture or management without recent hands-on coding or system implementation.",
        "5+ years experience in closed-source proprietary environments without external validation (open-source, talks, publications).",
    ],
    "context": [
        "Full-time Senior AI Engineer role based in Pune or Noida, India with hybrid work model and flexible office cadence.",
        "Expected experience range is 5–9 years, with ideal range around 6–8 years in applied machine learning or production AI roles.",
        "Role ownership includes ranking, retrieval, and candidate-job matching systems for a talent intelligence platform.",
        "Close collaboration with product managers, recruiters, and engineering teams to design and iterate ML systems in production.",
        "Fast-moving startup environment where systems evolve rapidly and ambiguity is expected.",
        "Expectation of strong ownership mindset, system thinking ability, and long-term commitment of at least 3+ years.",
    ],
}

### Embedding Job Description

In [14]:
print(type(jd_processed))
print(jd_processed)
jd_embedded = {
    section: embedder.embed(queries).tolist()
    for section, queries in tqdm(jd_processed.items(), desc="Embedding JD")
}

dump_json_file(jd_embedded, PROCESSED_FILENAMES["job_desc"])

<class 'dict'>
{'must_have': ['Senior AI Engineer role focused on production ML systems for ranking, retrieval, and candidate-job matching in a talent intelligence platform.', 'Strong production experience building embeddings-based retrieval systems using models like sentence-transformers, BGE, E5, or OpenAI embeddings, deployed in real user-facing systems.', 'Hands-on experience designing and operating vector search or hybrid retrieval systems using FAISS, Milvus, Weaviate, Pinecone, Elasticsearch, or OpenSearch.', 'Strong Python engineering skills with production-grade software development practices and ownership of deployed ML systems.', 'Experience designing and owning ranking evaluation frameworks including NDCG, MRR, MAP, offline-to-online evaluation, and A/B testing for production ranking systems.', 'Proven experience building and shipping end-to-end ranking, search, recommendation, or retrieval systems that are deployed to real users.', 'Experience owning production ML retrieva


Embedding JD:   0%|          | 0/7 [00:00<?, ?it/s]

[DEBUG]: Embedder.embed(...) called



Embedding JD: 100%|██████████| 7/7 [00:00<00:00, 22.20it/s]

[DEBUG]: Embedder.embed(...) returned in 0.289 s
[DEBUG]: Embedder.embed(...) called
[DEBUG]: Embedder.embed(...) returned in 0.006 s
[DEBUG]: Embedder.embed(...) called
[DEBUG]: Embedder.embed(...) returned in 0.003 s
[DEBUG]: Embedder.embed(...) called
[DEBUG]: Embedder.embed(...) returned in 0.003 s
[DEBUG]: Embedder.embed(...) called
[DEBUG]: Embedder.embed(...) returned in 0.003 s
[DEBUG]: Embedder.embed(...) called
[DEBUG]: Embedder.embed(...) returned in 0.004 s
[DEBUG]: Embedder.embed(...) called
[DEBUG]: Embedder.embed(...) returned in 0.003 s
[DEBUG]: dump_json_file(...) called
[DEBUG]: Dumping in "./processed/job_desc.json"
[DEBUG]: dump_json_file(...) finished in 0.018 s


## Pre-Processing Candidates

### Generate embed_data for all candidates

In [16]:
file = "/content/candidates.jsonl.gz"

vowels = "aeiouAEIOU"
with (
    __import__("gzip").open(file, "rb")
    if file.endswith(".gz")
    else open(file, "rb")
) as fp:
    embed_data = {
        c.candidate_id: c.embed_data
        for c in (msgspec.json.decode(line, type=Candidate) for line in fp)
    }

print(f"{len(embed_data) = }")

len(embed_data) = 100000


### Embedding Candidates data

#### Flattening embed_data and generating lookup tables

In [17]:
embed_data_flat = []

summary_lookup: List[Dict[Literal["cid", "idx"], str | int]] = []
skills_lookup: List[Dict[Literal["cid", "idx"], str | int]] = []
experience_lookup: List[Dict[Literal["cid", "idx"], str | int]] = []


# Flattenning embed_data
for cid, candidate in tqdm(embed_data.items(), desc="Flattening summary"):
    summary_lookup.append({"cid": cid, "idx": 0})
    embed_data_flat.append(candidate["summary"])

for cid, candidate in tqdm(embed_data.items(), desc="Flattening skill clusters"):
    for idx, skill in enumerate(candidate["skills"]):
        skills_lookup.append({"cid": cid, "idx": idx})
        embed_data_flat.append(skill)

for cid, candidate in tqdm(embed_data.items(), desc="Flattening experience"):
    for idx, exp in enumerate(candidate["experience"]):
        experience_lookup.append({"cid": cid, "idx": idx})
        embed_data_flat.append(exp)


# Save Lookup Tables
debug_print("Saving lookup tables")
dump_json_file(summary_lookup, PROCESSED_FILENAMES["lookup"]["summary"])
dump_json_file(skills_lookup, PROCESSED_FILENAMES["lookup"]["skills"])
dump_json_file(experience_lookup, PROCESSED_FILENAMES["lookup"]["experience"])


# Visual Verification
print()
print(f"{len(summary_lookup) = }")
print(f"{len(skills_lookup) = }")
print(f"{len(experience_lookup) = }")
print(f"{len(embed_data_flat) = }")

Flattening experience: 100%|██████████| 100000/100000 [00:00<00:00, 487756.84it/s]


[DEBUG]: Saving lookup tables
[DEBUG]: dump_json_file(...) called
[DEBUG]: Dumping in "./processed/lookup/summary.json"
[DEBUG]: dump_json_file(...) finished in 0.242 s
[DEBUG]: dump_json_file(...) called
[DEBUG]: Dumping in "./processed/lookup/skills.json"
[DEBUG]: dump_json_file(...) finished in 1.257 s
[DEBUG]: dump_json_file(...) called
[DEBUG]: Dumping in "./processed/lookup/experience.json"
[DEBUG]: dump_json_file(...) finished in 0.718 s

len(summary_lookup) = 100000
len(skills_lookup) = 541734
len(experience_lookup) = 300171
len(embed_data_flat) = 941905


#### Embedding Candidate's embed_data

In [18]:
debug_print("Bulk embedding flattened embed_datas")
embedded_data_flat = embedder.embed(embed_data_flat)


# Slicing into parts
debug_print("Slicing flat vectors into categories")
summary_vectors = embedded_data_flat[ : len(summary_lookup)]
skills_vectors = embedded_data_flat[len(summary_lookup) : len(summary_lookup) + len(skills_lookup)]
experience_vectors = embedded_data_flat[len(summary_lookup) + len(skills_lookup) : ]


# Save the vectors
debug_print("Saving summary vectors")
np.save(
    PROCESSED_FILENAMES["vectors"]["summary"],
    summary_vectors.astype(np.float16)
)

debug_print("Saving skills vectors")
np.save(
    PROCESSED_FILENAMES["vectors"]["skills"],
    skills_vectors.astype(np.float16)
)

debug_print("Saving experience vectors")
np.save(
    PROCESSED_FILENAMES["vectors"]["experience"],
    experience_vectors.astype(np.float16)
)


# Visual Verification
print()
print(f"{len(summary_vectors) = }")
print(f"{len(skills_vectors) = }")
print(f"{len(experience_vectors) = }")
print(f"{len(embedded_data_flat) = }")

[DEBUG]: Bulk embedding flattened embed_datas
[DEBUG]: Embedder.embed(...) called
[DEBUG]: Embedder.embed(...) returned in 590.967 s
[DEBUG]: Slicing flat vectors into categories
[DEBUG]: Saving summary vectors
[DEBUG]: Saving skills vectors
[DEBUG]: Saving experience vectors

len(summary_vectors) = 100000
len(skills_vectors) = 541734
len(experience_vectors) = 300171
len(embedded_data_flat) = 941905


In [19]:
print(len(embedded_data_flat))

941905


### Creating FAISS HNSW Flat Indexes for Candidate vectors

In [20]:
# Normalizing vectors
debug_print("Normalizing summary vectors")
faiss.normalize_L2(summary_vectors)

debug_print("Normalizing skills vectors")
faiss.normalize_L2(skills_vectors)

debug_print("Normalizing experience vectors")
faiss.normalize_L2(experience_vectors)


# Creating empty indexes
debug_print(f"Creating empty indices of {summary_vectors.shape[1]} dimensions")
# Adjust values (Higher = Higher Recall, Higher = Higher Memory + Dist usage)
summary_index = faiss.IndexHNSWFlat(summary_vectors.shape[1], 16) # Adjust from 16-24
skills_index = faiss.IndexHNSWFlat(skills_vectors.shape[1], 32) # Adjust from 32-40
experience_index = faiss.IndexHNSWFlat(experience_vectors.shape[1], 20) # Adjust from 20-28


# Adding Vectors
debug_print("Bulk inserting summary vectors to summary index.")
summary_index.add(summary_vectors)

debug_print("Bulk inserting skills vectors to skills index.")
skills_index.add(skills_vectors)

debug_print("Bulk inserting experience vectors to experience index.")
experience_index.add(experience_vectors)


# Saving indexes
debug_print("Saving summary index")
faiss.write_index(summary_index, PROCESSED_FILENAMES["indexes"]["summary"])

debug_print("Saving skills index")
faiss.write_index(skills_index, PROCESSED_FILENAMES["indexes"]["skills"])

debug_print("Saving experience index")
faiss.write_index(experience_index, PROCESSED_FILENAMES["indexes"]["experience"])

debug_print("Done")

[DEBUG]: Normalizing summary vectors
[DEBUG]: Normalizing skills vectors
[DEBUG]: Normalizing experience vectors
[DEBUG]: Creating empty indices of 384 dimensions
[DEBUG]: Bulk inserting summary vectors to summary index.
[DEBUG]: Bulk inserting skills vectors to skills index.
[DEBUG]: Bulk inserting experience vectors to experience index.
[DEBUG]: Saving summary index
[DEBUG]: Saving skills index
[DEBUG]: Saving experience index
[DEBUG]: Done


# Ranking
> **NOTE:** all files declared in **`PROCESSED_FILENAMES`** must exists and be valid before this.

## Resolving Dependencies

In [21]:
!pip install sentence_transformers

In [22]:
import csv
from math import ceil
from sentence_transformers import CrossEncoder

## Loading Data (Candidates, Indices, Vectors, JD)

In [40]:
# Loading Vectors
debug_print("Loading summary vectors ...")
summary_vectors = np.load(PROCESSED_FILENAMES["vectors"]["summary"])

debug_print("Loading skills vectors ...")
skills_vectors = np.load(PROCESSED_FILENAMES["vectors"]["skills"])

debug_print("Loading experience vectors ...")
experience_vectors = np.load(PROCESSED_FILENAMES["vectors"]["experience"])


# Loading Indices
debug_print("Loading summary index ...")
summary_index = faiss.read_index(PROCESSED_FILENAMES["indexes"]["summary"])

debug_print("Loading skills index ...")
skills_index = faiss.read_index(PROCESSED_FILENAMES["indexes"]["skills"])

debug_print("Loading experience vectors ...")
experience_index = faiss.read_index(PROCESSED_FILENAMES["indexes"]["experience"])


# Loading lookup tables
debug_print("Loading lookup tables")
summary_lookup = load_json_file(PROCESSED_FILENAMES["lookup"]["summary"])
skills_lookup = load_json_file(PROCESSED_FILENAMES["lookup"]["skills"])
experience_lookup = load_json_file(PROCESSED_FILENAMES["lookup"]["experience"])


# Loading JD
debug_print("Loading Processed Job Description")
job_desc = {
        section: np.array(vectors)
        for section, vectors in tqdm(load_json_file(
            PROCESSED_FILENAMES["job_desc"]
        ).items(), desc="Job Description")
    }


# Loading Candidates
debug_print("Loading candidates")
with (
    __import__("gzip").open(file, "rb")
    if file.endswith(".gz")
    else open(file, "rb")
) as fp:
    candidates = {
        c.candidate_id: {
            "candidate": c,
            "embedded": {
                "sources": c.embed_data,
                "vectors": {
                    "summary": None,
                    "skills": [],
                    "experience": [],
                }
            }
        }
        for c in (msgspec.json.decode(line, type=Candidate) for line in fp)
    }

# Populating candidates with vectors
debug_print("Populating candidates' summary with summary vectors")
for idx, entry in enumerate(tqdm(summary_lookup, desc="Summary")):
    candidates[entry["cid"]]["embedded"]["vectors"]["summary"] = summary_vectors[idx]

debug_print("Populating candidates' skills with skills vectors")
for idx, entry in enumerate(tqdm(skills_lookup, desc="Skills")):
    candidates[entry["cid"]]["embedded"]["vectors"]["skills"].append((skills_vectors[idx], entry["idx"]))

debug_print("Populating candidates' experience with experience vectors")
for idx, entry in enumerate(tqdm(experience_lookup, desc="Experience")):
    candidates[entry["cid"]]["embedded"]["vectors"]["experience"].append((experience_vectors[idx], entry["idx"]))

debug_print("Sorting skills and experience based on indices")
for c in tqdm(candidates.values(), desc="Sorting"):
    c["embedded"]["vectors"]["skills"] = list(map(lambda x: x[0], sorted(c["embedded"]["vectors"]["skills"], key=lambda x: x[1])))
    c["embedded"]["vectors"]["experience"] = list(map(lambda x: x[0], sorted(c["embedded"]["vectors"]["experience"], key=lambda x: x[1])))


# Visual Verification
print()
print(f"{len(candidates) = }")
print(f"{len(summary_vectors) = }")
print(f"{len(skills_vectors) = }")
print(f"{len(experience_vectors) = }")

[DEBUG]: Loading summary vectors ...
[DEBUG]: Loading skills vectors ...
[DEBUG]: Loading experience vectors ...
[DEBUG]: Loading summary index ...
[DEBUG]: Loading skills index ...
[DEBUG]: Loading experience vectors ...
[DEBUG]: Loading lookup tables
[DEBUG]: load_json_file(...) called
[DEBUG]: Loading from "./processed/lookup/summary.json"
[DEBUG]: load_json_file(...) returned in 0.104 s
[DEBUG]: load_json_file(...) called
[DEBUG]: Loading from "./processed/lookup/skills.json"
[DEBUG]: load_json_file(...) returned in 0.376 s
[DEBUG]: load_json_file(...) called
[DEBUG]: Loading from "./processed/lookup/experience.json"
[DEBUG]: load_json_file(...) returned in 0.157 s
[DEBUG]: Loading Processed Job Description
[DEBUG]: load_json_file(...) called
[DEBUG]: Loading from "./processed/job_desc.json"
[DEBUG]: load_json_file(...) returned in 0.006 s






Job Description: 100%|██████████| 7/7 [00:00<00:00, 1434.51it/s]

[DEBUG]: Loading candidates



Initial Scoring:   0%|          | 0/100000 [03:47<?, ?it/s]


[DEBUG]: Populating candidates' summary with summary vectors


...

[DEBUG]: Populating candidates' experience with experience vectors


Experience: 100%|██████████| 300171/300171 [00:00<00:00, 569875.57it/s]


[DEBUG]: Sorting skills and experience based on indices


Sorting: 100%|██████████| 100000/100000 [00:00<00:00, 157816.53it/s]


len(candidates) = 100000
len(summary_vectors) = 100000
len(skills_vectors) = 541734
len(experience_vectors) = 300171


## Initial Retrieval

In [45]:
TOP_K = 2000 # <<<------------------- change if needed

def retrieve(index, lookup, queries, k=2000):
    # Performing retrieval and accumulating hits for candidates
    D, I = index.search(queries, k)  # noqa: E741
    candi_hits = defaultdict(list)
    for row in range(len(queries)):
        for idx, distance in zip(I[row], D[row]):
            if idx == -1:
                continue
            cid = lookup[idx]["cid"]
            candi_hits[cid].append(float(distance))
    return candi_hits

retrieval_results = {
    section: {
        "summary": retrieve(summary_index, summary_lookup, queries, TOP_K),
        "skills": retrieve(skills_index, skills_lookup, queries, TOP_K),
        "experience": retrieve(experience_index, experience_lookup, queries, TOP_K),
    } for section, queries in tqdm(job_desc.items(), desc="Initial Retrieval")
}

# Store retrieval data in candidate wrappers for score_candidate to use
for section, result in retrieval_results.items():
    for category, candis in result.items():
        for cid, distances in candis.items():
            if "retrieval_data" not in candidates[cid]:
                candidates[cid]["retrieval_data"] = {}
            if category not in candidates[cid]["retrieval_data"]:
                candidates[cid]["retrieval_data"][category] = {}
            if section not in candidates[cid]["retrieval_data"][category]:
                candidates[cid]["retrieval_data"][category][section] = []
            candidates[cid]["retrieval_data"][category][section].extend(distances)


Initial Retrieval: 100%|██████████| 7/7 [00:00<00:00, 17.01it/s]


## Initial Scoring

In [46]:
def cosine_from_l2(distance):
    return 1 - (distance / 2)

def cosine_similarity(a, b):
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return (
        np.dot(a, b) / (norm_a * norm_b)
        if not (norm_a == 0 or norm_b == 0)
        else 0.0
    )

def normalize(v):
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

def centroid(vectors):
    return normalize(np.mean(vectors, axis=0))

def score_candidate(candi):
    candidate = candi["candidate"]
    data = candi.get("retrieval_data", {})

    skills_vecs = candi["embedded"]["vectors"]["skills"]
    exp_vecs = candi["embedded"]["vectors"]["experience"]
    summary_vec = candi["embedded"]["vectors"]["summary"]

    # Initialize reason features
    features = ReasonFeatures()
    features.yoe = candidate.profile.years_of_experience
    features.current_title = candidate.profile.current_title
    features.current_company = candidate.profile.current_company
    features.current_industry = candidate.profile.current_industry
    features.past_titles = [c.title for c in candidate.career_history[1:]]
    features.past_industries = list(
        set(c.industry for c in candidate.career_history[1:])
    )
    features.profile_completeness = candidate.redrob_signals.profile_completeness_score
    features.is_verified = (
        candidate.redrob_signals.verified_email
        and candidate.redrob_signals.verified_phone
    )
    features.is_active = candidate.redrob_signals.profile_views_received_30d > 10
    features.open_to_work = candidate.redrob_signals.open_to_work_flag
    features.response_rate = candidate.redrob_signals.recruiter_response_rate
    features.github_active = candidate.redrob_signals.github_activity_score > 0

    # Count skills
    features.relevant_skill_count = len(candidate.skills)
    features.advanced_skill_count = sum(
        1 for s in candidate.skills if s.proficiency == "advanced"
    )
    features.top_skills = [s.name for s in candidate.skills[:5]]

    assessments = candidate.redrob_signals.skill_assessment_scores
    if assessments:
        features.skill_assessment_avg = np.mean(list(assessments.values()))

    # Finding centroid of vectors
    skills_c = centroid(skills_vecs) if skills_vecs else None
    exp_c = centroid(exp_vecs) if exp_vecs else None

    # Initial Retrieval Score
    match_score = 0.0
    top_sections = []

    for category, sections in data.items():
        category_weight = CATEGORY_WEIGHTS.get(category, 1.0)

        for section, distances in sections.items():
            section_weight = SECTION_WEIGHTS.get(section, 1.0)

            similarities = list(
                sorted((cosine_from_l2(d) for d in distances), reverse=True)
            )
            signal = similarities[0]

            if len(similarities) > 1:
                signal += 0.25 * similarities[1]
            if len(similarities) > 2:
                signal += 0.25 * similarities[2]

            match_score += signal * category_weight * section_weight

            # Track top matching sections for reason
            if similarities:
                top_sections.append((section, similarities[0]))

    features.match_score = match_score
    features.top_matching_sections = [
        s[0] for s in sorted(top_sections, key=lambda x: x[1], reverse=True)[:3]
    ]

    # Determine match category
    if match_score >= 0.7:
        features.match_category = "high"
    elif match_score >= 0.4:
        features.match_category = "medium"
    else:
        features.match_category = "low"

    # Check if current/past roles are relevant (simplified check)
    features.has_relevant_current_role = features.match_category == "high"
    features.has_relevant_past_role = any(
        s[0] in ("experience", "skills") for s in top_sections[:3]
    )

    # Internal mismatch scoring
    penalty = 0.0
    penalty_breakdown = {}

    if skills_c is not None and exp_c is not None:
        similarity = cosine_similarity(skills_c, exp_c)
        p = (1 - similarity) * CANDIDATE_PENALTY["skills_experience_mismatch"]
        penalty += p
        penalty_breakdown["skills_experience_mismatch"] = p

    if summary_vec is not None and exp_c is not None:
        similarity = cosine_similarity(summary_vec, exp_c)
        p = (1 - similarity) * CANDIDATE_PENALTY["summary_experience_mismatch"]
        penalty += p
        penalty_breakdown["summary_experience_mismatch"] = p

    if summary_vec is not None and skills_c is not None:
        similarity = cosine_similarity(summary_vec, skills_c)
        p = (1 - similarity) * CANDIDATE_PENALTY["summary_skills_mismatch"]
        penalty += p
        penalty_breakdown["summary_skills_mismatch"] = p

    features.internal_penalty = penalty
    features.penalty_breakdown = penalty_breakdown

    # Analyze skill source
    skill_source, _ = candidate._analyze_skill_sources()
    features.skill_source = skill_source

    # Redrob signal matching
    redrob_score = 0.0
    for signal, weight in REDROB_SIGNAL_WEIGHTS.items():
        redrob_score += weight * getattr(candidate.redrob_signals, signal)

    total_score = (
        OVERALL_WEIGHTS["match"] * match_score
        + OVERALL_WEIGHTS["penalty"] * penalty
        + OVERALL_WEIGHTS["redrob"] * redrob_score
    )

    # Store features on candidate for reason generation
    candidate._reason_features = features

    return total_score


# Initial Scoring
candi_scores = dict(sorted((
    (cid, score_candidate(candidates[cid]))
    for cid in tqdm(candidates.keys(), desc="Initial Scoring")
), key=lambda x: x[1]))


Initial Scoring:   0%|          | 0/100000 [10:54<?, ?it/s]

Initial Scoring: 100%|██████████| 100000/100000 [00:30<00:00, 3281.57it/s]


## Reranking

### Download the Cross Encoder

In [49]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")

config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

### Running Cross Encoder Re-Ranker

In [55]:
@debug
def rerank(query: str, model, candis: List[Candidate]) -> Dict[CandidateId, float]:
    texts = (candi.text for candi in candis)
    pairs = [(query, t) for t in texts]

    scores = model.predict(pairs, batch_size=32)
    ranked_idx = np.argsort(scores)[::-1]
    return {candis[idx].candidate_id: float(scores[idx]) for idx in ranked_idx}

@debug
def build_jd_query(processed):
    # Building JD Query for re-ranking
    must_have = " ".join(processed["must_have"])
    core = " ".join(processed["core_ml_retrieval"] + processed["evaluation_ml_depth"])
    context = " ".join(processed["context"] + processed["execution_signal"])
    return (f"ROLE: \n{must_have}\n\nSKILLS: \n{core}\n\nCONTEXT: \n{context}").strip()

# Reranking
TOP_K = 200 # <<<------------------- Modify this
debug_print("Reranking ...")
reranker_scores = rerank(
    build_jd_query(jd_processed),
    CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2"),
    [candidates[cid]["candidate"] for cid in list(candi_scores.keys())[:TOP_K]],
)

# Accumulating
for cid, score in tqdm(reranker_scores.items(), desc="Accumulating reranker scores"):
    candi_scores[cid] += score * OVERALL_WEIGHTS["reranker"]

[DEBUG]: Reranking ...
[DEBUG]: build_jd_query(...) called
[DEBUG]: build_jd_query(...) returned in 0.000 s


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[DEBUG]: rerank(...) called
[DEBUG]: rerank(...) returned in 3.199 s



Accumulating reranker scores: 100%|██████████| 200/200 [00:00<00:00, 283590.53it/s]


## Final Ranking

In [56]:
import csv

# Normalizing
_max_score = ceil(max(list(candi_scores.values())))
normalized = {
    cid: round(score / _max_score, 4)
    for cid, score in tqdm(candi_scores.items(), desc="Normalizing scores")
}


# Final Sorting
debug_print("Final sorting")
ranked = sorted(normalized.items(), key=lambda x: (-x[1], x[0]), reverse=False)


# Saving
TOP_K = 100 # <<<------------------- Modify this
debug_print(f"Saving top {TOP_K}")
with open(OUTPUT_PATH, "w", newline="") as fp:
    writer = csv.writer(fp)
    writer.writerow(("candidate_id", "rank", "score", "reasoning"))
    writer.writerows(
        (cid, rank, score, candidates[cid]["candidate"].reason)
        for rank, (cid, score) in enumerate(ranked[:100], 1)
    )


Normalizing scores: 100%|██████████| 100000/100000 [00:00<00:00, 160404.77it/s]


[DEBUG]: Final sorting
[DEBUG]: Saving top 100
